In [1]:
import numpy as np
import pandas as pd
import json
import pickle
import os

from scipy.sparse import load_npz

# Charger la matrice TF-IDF
tfidf_matrix = load_npz(
    "../data/processed/tfidf_matrix.npz"
)

# Charger la matrice des interactions
user_item = load_npz(
    "../data/processed/interaction_matrix.npz"
).tocsr()

# Charger les mappings
with open("../data/processed/mappings.json", "r") as f:
    mappings = json.load(f)

user_to_idx = mappings["user_to_idx"]
item_to_idx = mappings["item_to_idx"]

idx_to_item = {v: k for k, v in item_to_idx.items()}

# Charger le catalogue des articles
news_catalog = pd.read_csv(
    "../data/processed/news_catalog.csv"
)

# Charger les interactions d'entraînement
train_interactions = pd.read_csv(
    "../data/processed/train_interactions.csv"
)

# Charger les impressions de validation
dev_impressions = pd.read_csv(
    "../data/processed/dev_impressions.csv.gz"
)

# Charger l'historique d'entraînement
with open("../data/processed/train_history.pkl", "rb") as f:
    train_history = pickle.load(f)

print("Données chargées avec succès !")
print("Matrice TF-IDF :", tfidf_matrix.shape)
print("Matrice interactions :", user_item.shape)
print("Catalogue :", news_catalog.shape)
print("Interactions train :", train_interactions.shape)
print("Impressions validation :", dev_impressions.shape)
print("Nombre d'utilisateurs :", len(user_to_idx))
print("Nombre d'articles :", len(item_to_idx))

Données chargées avec succès !
Matrice TF-IDF : (65238, 20000)
Matrice interactions : (50000, 65238)
Catalogue : (65238, 9)
Interactions train : (234468, 2)
Impressions validation : (2740998, 6)
Nombre d'utilisateurs : 50000
Nombre d'articles : 65238


In [2]:
import os

for file in os.listdir("../data/processed/"):
    if file.endswith(".pkl"):
        print(file)

dev_history.pkl
tfidf_vectorizer.pkl
train_history.pkl


In [4]:
import pickle

# Charger l'historique d'entraînement
with open("../data/processed/train_history.pkl", "rb") as f:
    train_history = pickle.load(f)

# Charger l'historique de validation
with open("../data/processed/dev_history.pkl", "rb") as f:
    dev_history = pickle.load(f)

print("Train history :", type(train_history), len(train_history))
print("Dev history :", type(dev_history), len(dev_history))
print("TF-IDF matrix :", tfidf_matrix.shape)

Train history : <class 'pandas.DataFrame'> 156965
Dev history : <class 'pandas.DataFrame'> 73152
TF-IDF matrix : (65238, 20000)


In [5]:
print("Exemple train_history :")
print(list(train_history.items())[:1])

print("\nExemple dev_history :")
print(list(dev_history.items())[:1])

Exemple train_history :
[('impression_id', 0              1
1              2
2              3
3              4
4              5
           ...  
156960    156961
156961    156962
156962    156963
156963    156964
156964    156965
Name: impression_id, Length: 156965, dtype: int64)]

Exemple dev_history :
[('impression_id', 0            1
1            2
2            3
3            4
4            5
         ...  
73147    73148
73148    73149
73149    73150
73150    73151
73151    73152
Name: impression_id, Length: 73152, dtype: int64)]


In [6]:
print("Type :", type(train_history))

if isinstance(train_history, dict):
    print("Clés :", train_history.keys())

    for key, value in train_history.items():
        print(f"\nClé : {key}")
        print("Type :", type(value))
        print("Taille :", len(value))

elif isinstance(train_history, pd.DataFrame):
    print("Colonnes :", train_history.columns.tolist())
    display(train_history.head())

Type : <class 'pandas.DataFrame'>
Colonnes : ['impression_id', 'user_id', 'history', 'history_list', 'history_length', 'history_indices']


,impression_id,user_id,history,history_list,history_length,history_indices
0,1,U13740,N55189 N42782 N34694 N45794 N18445 N63302 N104...,"[N55189, N42782, N34694, N45794, N18445, N6330...",9,"[6892, 10049, 15555, 21466, 26357, 4945, 14070..."
1,2,U91836,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,"[N31739, N6072, N63045, N23979, N35656, N43353...",82,"[25815, 2333, 8523, 12086, 13462, 14201, 12732..."
2,3,U73700,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,"[N10732, N25792, N7563, N21087, N41087, N5445,...",16,"[5476, 4206, 11683, 7703, 8123, 23393, 22969, ..."
3,4,U34670,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,"[N45729, N2203, N871, N53880, N41375, N43142, ...",10,"[13826, 19084, 28505, 7023, 22909, 16666, 1559..."
4,5,U8125,N10078 N56514 N14904 N33740,"[N10078, N56514, N14904, N33740]",4,"[23642, 4852, 27685, 31188]"


In [7]:
print("Nombre de lignes :", len(train_history))
print("Nombre d'utilisateurs uniques :", train_history["user_id"].nunique())

print("\nExemple d'historique :")
print(train_history.iloc[0]["history_list"])

print("\nIndices TF-IDF :")
print(train_history.iloc[0]["history_indices"])

print("\nNombre d'indices :", len(train_history.iloc[0]["history_indices"]))

print("\nVérification des dimensions :")
print("Matrice TF-IDF :", tfidf_matrix.shape)
print("Nombre d'articles dans le mapping :", len(item_to_idx))

Nombre de lignes : 156965
Nombre d'utilisateurs uniques : 50000

Exemple d'historique :
['N55189', 'N42782', 'N34694', 'N45794', 'N18445', 'N63302', 'N10414', 'N19347', 'N31801']

Indices TF-IDF :
[6892, 10049, 15555, 21466, 26357, 4945, 14070, 13910, 2077]

Nombre d'indices : 9

Vérification des dimensions :
Matrice TF-IDF : (65238, 20000)
Nombre d'articles dans le mapping : 65238


In [8]:
# Vérifier la correspondance pour les 5 premiers historiques

for i in range(5):
    row = train_history.iloc[i]

    expected_indices = [
        item_to_idx[news_id]
        for news_id in row["history_list"]
        if news_id in item_to_idx
    ]

    print(f"Historique {i + 1}")
    print("Indices enregistrés :", row["history_indices"])
    print("Indices du mapping  :", expected_indices)
    print(
        "Correspondance :",
        list(row["history_indices"]) == expected_indices
    )
    print("-" * 50)

Historique 1
Indices enregistrés : [6892, 10049, 15555, 21466, 26357, 4945, 14070, 13910, 2077]
Indices du mapping  : [6892, 10049, 15555, 21466, 26357, 4945, 14070, 13910, 2077]
Correspondance : True
--------------------------------------------------
Historique 2
Indices enregistrés : [25815, 2333, 8523, 12086, 13462, 14201, 12732, 26997, 22953, 21716, 28043, 2393, 28919, 23106, 17163, 15870, 25724, 28191, 29417, 17079, 28645, 28582, 17679, 30634, 14856, 8177, 2756, 4500, 2599, 26143, 22036, 4852, 10969, 128, 19187, 5641, 25936, 22637, 10692, 27867, 13918, 13994, 1081, 26637, 25905, 20528, 6874, 28124, 1536, 29990, 18319, 22198, 15517, 31737, 22534, 29497, 27721, 28380, 29786, 6014, 22967, 8645, 26035, 26487, 10805, 6014, 25316, 19537, 19073, 19940, 8547, 26923, 28469, 29481, 14580, 8055, 21078, 13644, 32017, 33752, 40661, 34839]
Indices du mapping  : [25815, 2333, 8523, 12086, 13462, 14201, 12732, 26997, 22953, 21716, 28043, 2393, 28919, 23106, 17163, 15870, 25724, 28191, 29417, 1707

In [9]:
from sklearn.preprocessing import normalize

# Construire les profils utilisateurs
user_profiles = user_item @ tfidf_matrix

# Normaliser les profils pour utiliser la similarité cosinus
user_profiles = normalize(user_profiles, norm="l2", axis=1)

# Normaliser également les vecteurs des articles
tfidf_matrix = normalize(tfidf_matrix, norm="l2", axis=1)

print("Profils Content-Based construits !")
print("Dimensions des profils :", user_profiles.shape)
print("Dimensions des articles :", tfidf_matrix.shape)

# Vérifier un utilisateur
user_idx = 0

print("\nUtilisateur :", user_idx)
print("Nombre d'articles consultés :", user_item[user_idx].nnz)
print("Nombre de caractéristiques du profil :", user_profiles[user_idx].nnz)

Profils Content-Based construits !
Dimensions des profils : (50000, 20000)
Dimensions des articles : (65238, 20000)

Utilisateur : 0
Nombre d'articles consultés : 1
Nombre de caractéristiques du profil : 49


In [10]:
from implicit.als import AlternatingLeastSquares
import time

als_model = AlternatingLeastSquares(
    factors=50,
    regularization=0.1,
    iterations=15,
    random_state=42
)

start_time = time.perf_counter()

als_model.fit(user_item * 20)

training_time = time.perf_counter() - start_time

print("Modèle ALS entraîné !")
print("User factors :", als_model.user_factors.shape)
print("Item factors :", als_model.item_factors.shape)
print("Temps d'entraînement :", round(training_time, 2), "secondes")

c:\Users\MSI\AppData\Local\Programs\Python\Python313\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

Modèle ALS entraîné !
User factors : (50000, 50)
Item factors : (65238, 50)
Temps d'entraînement : 3.02 secondes


In [11]:
# Garder les impressions des utilisateurs connus par ALS
known_dev = dev_impressions[
    dev_impressions["user_id"].isin(user_to_idx.keys())
].copy()

# Ajouter les indices des utilisateurs
known_dev["user_idx"] = known_dev["user_id"].map(user_to_idx)

# Ajouter les indices des articles
known_dev["item_idx"] = known_dev["news_id"].map(item_to_idx)

# Vérifier les valeurs manquantes
print("Nombre d'impressions :", len(known_dev))
print("Nombre d'utilisateurs :", known_dev["user_id"].nunique())

print("Utilisateurs sans indice :", known_dev["user_idx"].isna().sum())
print("Articles sans indice :", known_dev["item_idx"].isna().sum())

print("\nNombre de clics :", known_dev["label"].sum())

Nombre d'impressions : 337254
Nombre d'utilisateurs : 5943
Utilisateurs sans indice : 0
Articles sans indice : 0

Nombre de clics : 13593


In [12]:
# Récupérer les indices des utilisateurs et des articles
user_indices = known_dev["user_idx"].to_numpy(dtype=int)
item_indices = known_dev["item_idx"].to_numpy(dtype=int)

# 1. Scores Content-Based
cb_scores = user_profiles[user_indices].multiply(
    tfidf_matrix[item_indices]
).sum(axis=1)

known_dev["cb_score"] = np.asarray(cb_scores).ravel()

# 2. Scores ALS
user_vectors = als_model.user_factors[user_indices]
item_vectors = als_model.item_factors[item_indices]

known_dev["als_score"] = np.einsum(
    "ij,ij->i",
    user_vectors,
    item_vectors
)

# Afficher quelques résultats
display(
    known_dev[
        ["user_id", "impression_id", "news_id",
         "label", "cb_score", "als_score"]
    ].head(10)
)

print("\nVérification des scores :")
print(known_dev[["cb_score", "als_score"]].describe())

,user_id,impression_id,news_id,label,cb_score,als_score
698,U44035,24,N37204,1,0.035717,0.242400
699,U44035,24,N48487,0,0.049217,0.047931
700,U44035,24,N59933,0,0.032923,0.000000
701,U44035,24,N512,0,0.006305,0.000000
702,U44035,24,N51776,0,0.006411,0.000000
703,U44035,24,N64077,0,0.027686,0.000000
704,U44035,24,N4667,0,0.019863,0.000000
705,U44035,24,N56080,0,0.000000,0.000000
706,U44035,24,N496,1,0.047616,0.229896
707,U44035,24,N52492,0,0.094185,0.000000



Vérification des scores :
            cb_score      als_score
count  337254.000000  337254.000000
mean        0.019358       0.011506
std         0.035433       0.071389
min         0.000000      -1.345099
25%         0.000000       0.000000
50%         0.008584       0.000000
75%         0.024265       0.001170
max         1.000000       1.680561


In [13]:
def min_max_normalize(scores):
    min_score = scores.min()
    max_score = scores.max()

    if max_score == min_score:
        return np.zeros(len(scores))

    return (scores - min_score) / (max_score - min_score)


# Normaliser Content-Based
known_dev["cb_norm"] = (
    known_dev.groupby(["user_id", "impression_id"])["cb_score"]
    .transform(min_max_normalize)
)

# Normaliser ALS
known_dev["als_norm"] = (
    known_dev.groupby(["user_id", "impression_id"])["als_score"]
    .transform(min_max_normalize)
)

# Vérifier les résultats
print("Scores normalisés :")

display(
    known_dev[
        ["user_id", "impression_id", "news_id",
         "cb_norm", "als_norm"]
    ].head(10)
)

print("\nStatistiques après normalisation :")
print(known_dev[["cb_norm", "als_norm"]].describe())

Scores normalisés :


,user_id,impression_id,news_id,cb_norm,als_norm
698,U44035,24,N37204,0.311338,1.000000
699,U44035,24,N48487,0.429022,0.326187
700,U44035,24,N59933,0.286990,0.160111
701,U44035,24,N512,0.054957,0.160111
702,U44035,24,N51776,0.055881,0.160111
703,U44035,24,N64077,0.241340,0.160111
704,U44035,24,N4667,0.173142,0.160111
705,U44035,24,N56080,0.000000,0.160111
706,U44035,24,N496,0.415063,0.956675
707,U44035,24,N52492,0.821003,0.160111



Statistiques après normalisation :
             cb_norm       als_norm
count  337254.000000  337254.000000
mean        0.148904       0.386977
std         0.229035       0.256414
min         0.000000       0.000000
25%         0.000000       0.198246
50%         0.054063       0.359869
75%         0.188759       0.538897
max         1.000000       1.000000


In [14]:
# Reproduire le nettoyage utilisé lors de l'évaluation ALS

train_pairs = pd.MultiIndex.from_frame(
    train_interactions[["user_id", "news_id"]].drop_duplicates()
)

dev_pairs = pd.MultiIndex.from_frame(
    known_dev[["user_id", "news_id"]]
)

in_train = dev_pairs.isin(train_pairs)

clean_dev = known_dev.loc[
    ~((known_dev["label"].to_numpy() == 1) & in_train)
].copy()

print("Impressions avant nettoyage :", len(known_dev))
print("Impressions après nettoyage :", len(clean_dev))
print("Clics conservés :", clean_dev["label"].sum())

Impressions avant nettoyage : 337254
Impressions après nettoyage : 337228
Clics conservés : 13567


In [15]:
def evaluate_impression(group, score_column, k=10):

    group = group.sort_values(
        score_column,
        ascending=False
    )

    top_labels = group["label"].to_numpy()[:k]

    hits = top_labels.sum()
    total_relevant = group["label"].sum()

    precision = hits / k

    recall = (
        hits / total_relevant
        if total_relevant > 0 else np.nan
    )

    discounts = np.log2(np.arange(2, len(top_labels) + 2))

    dcg = np.sum(top_labels / discounts)

    ideal_labels = np.sort(
        group["label"].to_numpy()
    )[::-1][:k]

    ideal_dcg = np.sum(ideal_labels / discounts)

    ndcg = (
        dcg / ideal_dcg
        if ideal_dcg > 0 else np.nan
    )

    return pd.Series({
        "Precision@10": precision,
        "Recall@10": recall,
        "NDCG@10": ndcg
    })


alphas = [0.2, 0.4, 0.5, 0.6, 0.8]

hybrid_results = []

for alpha in alphas:

    # Calcul du score hybride
    clean_dev["hybrid_score"] = (
        alpha * clean_dev["cb_norm"]
        + (1 - alpha) * clean_dev["als_norm"]
    )

    # Évaluation de chaque impression
    metrics = clean_dev.groupby(
        ["user_id", "impression_id"]
    ).apply(
        lambda group: evaluate_impression(
            group, "hybrid_score"
        )
    )

    # Moyenne des métriques
    mean_metrics = metrics.mean()

    hybrid_results.append({
        "Alpha": alpha,
        "Precision@10": mean_metrics["Precision@10"],
        "Recall@10": mean_metrics["Recall@10"],
        "NDCG@10": mean_metrics["NDCG@10"]
    })

# Tableau comparatif
hybrid_results_df = pd.DataFrame(hybrid_results)

display(hybrid_results_df.round(4))

,Alpha,Precision@10,Recall@10,NDCG@10
0,0.2,0.0752,0.5855,0.3354
1,0.4,0.0760,0.5891,0.3438
2,0.5,0.0764,0.5912,0.3491
3,0.6,0.0767,0.5929,0.3512
4,0.8,0.0773,0.5964,0.3528


In [16]:
# Évaluer Content-Based seul
cb_metrics = clean_dev.groupby(
    ["user_id", "impression_id"]
).apply(
    lambda group: evaluate_impression(group, "cb_norm")
).mean()

# Évaluer ALS seul
als_metrics = clean_dev.groupby(
    ["user_id", "impression_id"]
).apply(
    lambda group: evaluate_impression(group, "als_norm")
).mean()

# Comparaison des trois modèles
comparison = pd.DataFrame([
    {
        "Model": "Content-Based",
        **cb_metrics.to_dict()
    },
    {
        "Model": "ALS",
        **als_metrics.to_dict()
    },
    {
        "Model": "Hybrid (alpha=0.8)",
        **hybrid_results_df.loc[
            hybrid_results_df["Alpha"] == 0.8,
            ["Precision@10", "Recall@10", "NDCG@10"]
        ].iloc[0].to_dict()
    }
])

display(comparison.round(4))

,Model,Precision@10,Recall@10,NDCG@10
0,Content-Based,0.0769,0.5931,0.3510
1,ALS,0.0683,0.5407,0.3091
2,Hybrid (alpha=0.8),0.0773,0.5964,0.3528


In [17]:
# Nombre d'interactions par article
item_interaction_counts = np.asarray(
    user_item.getnnz(axis=0)
).ravel()

# Articles avec au moins une interaction
known_items = np.where(item_interaction_counts > 0)[0]

# Articles sans aucune interaction
cold_items = np.where(item_interaction_counts == 0)[0]

print("Nombre total d'articles :", user_item.shape[1])
print("Articles avec interactions :", len(known_items))
print("Articles sans interactions :", len(cold_items))
print(
    "Pourcentage d'articles sans interactions :",
    round(len(cold_items) / user_item.shape[1] * 100, 2),
    "%"
)

Nombre total d'articles : 65238
Articles avec interactions : 7713
Articles sans interactions : 57525
Pourcentage d'articles sans interactions : 88.18 %


In [18]:
# Calculer la popularité des articles
popularity_df = pd.DataFrame({
    "news_id": [idx_to_item[i] for i in range(len(idx_to_item))],
    "popularity": item_interaction_counts
})

# Trier les articles du plus populaire au moins populaire
popularity_df = popularity_df.sort_values(
    by="popularity",
    ascending=False
).reset_index(drop=True)

print("Top 10 des articles les plus populaires :")
display(popularity_df.head(10))

Top 10 des articles les plus populaires :


,news_id,popularity
0,N55689,4257
1,N35729,3281
2,N33619,3214
3,N53585,2824
4,N63970,2550
5,N49685,2273
6,N49279,2236
7,N287,2111
8,N23446,1907
9,N51048,1861


In [19]:

# Construire le classement de popularité des articles

popularity_df = pd.DataFrame({
    "news_id": [idx_to_item[i] for i in range(len(idx_to_item))],
    "popularity": item_interaction_counts
})

popularity_df = popularity_df.sort_values(
    by="popularity",
    ascending=False
).reset_index(drop=True)

# Afficher les 10 articles les plus populaires
popularity_df.head(10)


,news_id,popularity
0,N55689,4257
1,N35729,3281
2,N33619,3214
3,N53585,2824
4,N63970,2550
5,N49685,2273
6,N49279,2236
7,N287,2111
8,N23446,1907
9,N51048,1861


In [20]:

# Vérifier les colonnes du catalogue
print("Colonnes :", news_catalog.columns.tolist())

# Vérifier les informations d'un article populaire
news_catalog[
    news_catalog["news_id"].isin(["N55689", "N35729", "N33619"])
][["news_id", "title", "category"]]


Colonnes : ['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities', 'content']


,news_id,title,category
33899,N55689,"Charles Rogers, former Michigan State football...",sports
41550,N33619,College gymnast dies following training accide...,news
50688,N35729,Porsche launches into second story of New Jers...,news


In [21]:
print("Exemple user_id :", list(user_to_idx.keys())[:5])
print("Type user_id :", type(list(user_to_idx.keys())[0]))

print("Exemple news_id :", list(item_to_idx.keys())[:5])
print("Type news_id :", type(list(item_to_idx.keys())[0]))

Exemple user_id : ['U100', 'U1000', 'U10001', 'U10003', 'U10008']
Type user_id : <class 'str'>
Exemple news_id : ['N55528', 'N19639', 'N61837', 'N53526', 'N38324']
Type news_id : <class 'str'>


In [26]:

def recommend_news(user_id, top_n=10, alpha=0.8):

    # 1. Cas Cold Start : utilisateur inconnu
    if user_id not in user_to_idx:
        result = popularity_df.head(top_n).merge(
            news_catalog[["news_id", "title", "category"]],
            on="news_id",
            how="left"
        )

        result["strategy"] = "Popularity-Based"
        result = result.rename(columns={"popularity": "score"})

        return result[
            ["news_id", "title", "category", "score", "strategy"]
        ]

    user_idx = user_to_idx[user_id]

    # 2. Cas utilisateur connu mais sans historique
    if user_item[user_idx].nnz == 0:
        result = popularity_df.head(top_n).merge(
            news_catalog[["news_id", "title", "category"]],
            on="news_id",
            how="left"
        )

        result["strategy"] = "Popularity-Based"

        result = result.rename(columns={"popularity": "score"})

        return result[
            ["news_id", "title", "category", "score", "strategy"]
        ]

    # 3. Récupérer les articles non encore consultés
    seen_items = user_item[user_idx].indices

    candidate_indices = np.setdiff1d(
        np.arange(len(item_to_idx)),
        seen_items
    )

    # 4. Calcul des scores Content-Based
    user_profile = user_profiles[user_idx]

    cb_scores = user_profile @ tfidf_matrix[candidate_indices].T

    # Convertir la matrice sparse en tableau NumPy dense
    cb_scores = cb_scores.toarray().ravel()

    cb_norm = min_max_normalize(cb_scores)

    # 5. Identifier les articles avec interactions
    has_interactions = (
        item_interaction_counts[candidate_indices] > 0
    )

    als_scores = np.full(len(candidate_indices), np.nan)

    # 6. Calcul ALS uniquement pour les articles connus
    warm_positions = np.where(has_interactions)[0]

    if len(warm_positions) > 0:

        warm_items = candidate_indices[warm_positions]

        user_vector = als_model.user_factors[user_idx]
        item_vectors = als_model.item_factors[warm_items]

        warm_scores = item_vectors @ user_vector

        als_norm = min_max_normalize(warm_scores)

        als_scores[warm_positions] = warm_scores

        # 7. Score hybride pour les articles connus
        final_scores = cb_norm.copy()

        final_scores[warm_positions] = (
            alpha * cb_norm[warm_positions]
            + (1 - alpha) * als_norm
        )

    else:
        # Aucun article candidat avec interactions
        final_scores = cb_norm.copy()

    # 8. Construire le résultat
    recommendations = pd.DataFrame({
        "news_id": [idx_to_item[i] for i in candidate_indices],
        "score": final_scores,
        "score_cb": cb_norm,
        "score_als": als_scores,
        "strategy": np.where(
            has_interactions,
            "Hybrid CB + ALS",
            "Content-Based (Cold Start Article)"
        )
    })

    # 9. Ajouter les titres et catégories
    recommendations = recommendations.merge(
        news_catalog[["news_id", "title", "category"]],
        on="news_id",
        how="left"
    )

    recommendations = recommendations.sort_values(
        "score",
        ascending=False
    ).head(top_n)

    return recommendations[
        ["news_id", "title", "category", "score",
         "score_cb", "score_als", "strategy"]
    ].reset_index(drop=True)


In [27]:
# Trouver un utilisateur connu avec un historique
test_user = next(
    user_id
    for user_id, user_idx in user_to_idx.items()
    if user_item[user_idx].nnz > 0
)

print("Utilisateur testé :", test_user)

recommendations = recommend_news(
    user_id=test_user,
    top_n=10
)

recommendations

Utilisateur testé : U100


,news_id,title,category,score,score_cb,score_als,strategy
0,N59145,Man shot multiple times during family's Hallow...,news,1.000000,1.000000,NaN,Content-Based (Cold Start Article)
1,N45079,"Sheriff's Office: Man killed wife, her sister ...",news,0.900621,0.900621,NaN,Content-Based (Cold Start Article)
2,N30228,A Texas man shot and killed his wife. He says ...,news,0.849000,0.849000,NaN,Content-Based (Cold Start Article)
3,N48573,Man kidnapped and left in Orange County storag...,news,0.828688,0.828688,NaN,Content-Based (Cold Start Article)
4,N23225,Estranged husband shoots wife at Carolina club...,news,0.821466,0.821466,NaN,Content-Based (Cold Start Article)
5,N44889,"Man arrested after kidnapping, shooting wife i...",news,0.806590,0.806590,NaN,Content-Based (Cold Start Article)
6,N39291,"Man kept mother's, aunt's bodies wrapped in ro...",news,0.791444,0.791444,NaN,Content-Based (Cold Start Article)
7,N44910,Life Flight transports 2 after major crash in ...,news,0.757952,0.830380,0.00333,Hybrid CB + ALS
8,N46913,Man found fatally shot in head in Brooklyn: po...,news,0.745829,0.745829,NaN,Content-Based (Cold Start Article)
9,N1308,Sheriff: Zearing man told neighbor he killed h...,news,0.744710,0.744710,NaN,Content-Based (Cold Start Article)


In [28]:
unknown_recommendations = recommend_news(
    user_id="U_UNKNOWN",
    top_n=10
)

unknown_recommendations

,news_id,title,category,score,strategy
0,N55689,"Charles Rogers, former Michigan State football...",sports,4257,Popularity-Based
1,N35729,Porsche launches into second story of New Jers...,news,3281,Popularity-Based
2,N33619,College gymnast dies following training accide...,news,3214,Popularity-Based
3,N53585,"Rip Taylor's Cause of Death Revealed, Memorial...",tv,2824,Popularity-Based
4,N63970,Dean Foods files for bankruptcy,finance,2550,Popularity-Based
5,N49685,Broadway Star Laurel Griggs Suffered Asthma At...,music,2273,Popularity-Based
6,N49279,Broadway Actress Laurel Griggs Dies at Age 13,music,2236,Popularity-Based
7,N287,Three school workers charged in death of speci...,news,2111,Popularity-Based
8,N23446,Prince Harry and Meghan Markle just shared a n...,lifestyle,1907,Popularity-Based
9,N51048,Rep. Tim Ryan endorses Biden in Democratic pri...,news,1861,Popularity-Based


In [29]:
users_without_history = [
    user_id
    for user_id, user_idx in user_to_idx.items()
    if user_item[user_idx].nnz == 0
]

print("Nombre d'utilisateurs sans historique :", len(users_without_history))

Nombre d'utilisateurs sans historique : 0


In [30]:

# Définir les trois modèles à comparer
alpha = 0.8

clean_dev["hybrid_score"] = (
    alpha * clean_dev["cb_norm"]
    + (1 - alpha) * clean_dev["als_norm"]
)

models = {
    "Content-Based": "cb_norm",
    "ALS": "als_norm",
    "Hybrid CB + ALS": "hybrid_score"
}

results = []

for model_name, score_column in models.items():

    # Calcul des métriques par impression
    metrics = clean_dev.groupby(
        ["user_id", "impression_id"]
    ).apply(
        lambda group: evaluate_impression(
            group, score_column, k=10
        )
    )

    # Moyenne des métriques
    results.append({
        "Model": model_name,
        "Precision@10": metrics["Precision@10"].mean(),
        "Recall@10": metrics["Recall@10"].mean(),
        "NDCG@10": metrics["NDCG@10"].mean()
    })

comparison_df = pd.DataFrame(results)

comparison_df.round(4)


,Model,Precision@10,Recall@10,NDCG@10
0,Content-Based,0.0769,0.5931,0.3510
1,ALS,0.0683,0.5407,0.3091
2,Hybrid CB + ALS,0.0773,0.5964,0.3528


In [31]:

import os
import json
import joblib
import numpy as np
from scipy.sparse import save_npz

# Créer le dossier des artefacts
ARTIFACTS_DIR = "../streamlit_app/artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# 1. Sauvegarder le modèle ALS
joblib.dump(
    als_model,
    os.path.join(ARTIFACTS_DIR, "als_model.pkl")
)

# 2. Sauvegarder les matrices
save_npz(
    os.path.join(ARTIFACTS_DIR, "user_item.npz"),
    user_item
)

save_npz(
    os.path.join(ARTIFACTS_DIR, "tfidf_matrix.npz"),
    tfidf_matrix
)

save_npz(
    os.path.join(ARTIFACTS_DIR, "user_profiles.npz"),
    user_profiles
)

# 3. Sauvegarder les mappings
with open(
    os.path.join(ARTIFACTS_DIR, "mappings.json"),
    "w"
) as f:
    json.dump({
        "user_to_idx": user_to_idx,
        "item_to_idx": item_to_idx
    }, f)

# 4. Sauvegarder le catalogue
news_catalog.to_csv(
    os.path.join(ARTIFACTS_DIR, "news_catalog.csv"),
    index=False
)

# 5. Sauvegarder la popularité
popularity_df.to_csv(
    os.path.join(ARTIFACTS_DIR, "popularity.csv"),
    index=False
)

# 6. Sauvegarder les métriques
comparison_df.to_csv(
    os.path.join(ARTIFACTS_DIR, "model_metrics.csv"),
    index=False
)

# 7. Sauvegarder le nombre d'interactions par article
np.save(
    os.path.join(ARTIFACTS_DIR, "item_interaction_counts.npy"),
    item_interaction_counts
)

print("Sauvegarde terminée !")
print("Nombre de fichiers :", len(os.listdir(ARTIFACTS_DIR)))
print("Fichiers :", os.listdir(ARTIFACTS_DIR))


Sauvegarde terminée !
Nombre de fichiers : 9
Fichiers : ['als_model.pkl', 'item_interaction_counts.npy', 'mappings.json', 'model_metrics.csv', 'news_catalog.csv', 'popularity.csv', 'tfidf_matrix.npz', 'user_item.npz', 'user_profiles.npz']
